In [ ]:
import numpy as np
import tensorflow as tf
from sklearn.datasets import make_blobs

# --- PASO 1: Crear datos falsos (Dataset) ---
# Generamos 100 puntos agrupados en 2 nubes separadas (Clase 0 y Clase 1)
# X son las coordenadas, y es la etiqueta (0 o 1)
X, y = make_blobs(n_samples=100, centers=2, n_features=2, random_state=42)

# --- PASO 2: Construir el Modelo (La parte importante) ---
model = tf.keras.Sequential([
    # Capa Densa con EXACTAMENTE 2 neuronas.
    # input_shape=(2,) significa que recibe coordenadas (x, y).
    # activation='softmax' hace que la suma de las 2 neuronas sea siempre 1 (100%).
    tf.keras.layers.Dense(2, input_shape=(2,), activation='softmax')
])

# --- PASO 3: Compilar ---
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# --- PASO 4: Entrenar ---
print("Entrenando el modelo...")
model.fit(X, y, epochs=20, verbose=0) # verbose=0 para no llenar la pantalla
print("¡Entrenamiento completado!\n")

# --- PASO 5: Probar con un dato nuevo ---
nuevo_dato = np.array([[ -2, 8 ]]) # Un punto de ejemplo
prediccion = model.predict(nuevo_dato)

print("--- RESULTADOS ---")
print(f"Dato de entrada: {nuevo_dato}")
print(f"Activación Neurona 1 (Probabilidad Clase 0): {prediccion[0][0]:.4f}")
print(f"Activación Neurona 2 (Probabilidad Clase 1): {prediccion[0][1]:.4f}")

clase_ganadora = np.argmax(prediccion)
print(f"--> El modelo dice que pertenece a la Clase: {clase_ganadora}")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Entrenando el modelo...
¡Entrenamiento completado!

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step
--- RESULTADOS ---
Dato de entrada: [[-2  8]]
Activación Neurona 1 (Probabilidad Clase 0): 0.0001
Activación Neurona 2 (Probabilidad Clase 1): 0.9999
--> El modelo dice que pertenece a la Clase: 1


In [ ]:
import numpy as np
import tensorflow as tf

# --- PASO 1: Preparar los datos (Secuencias) ---
# Queremos que aprenda que después de X viene X + 10
# Datos de entrada (X): Secuencias de 3 números
X_train = np.array([
    [10, 20, 30],
    [20, 30, 40],
    [30, 40, 50],
    [40, 50, 60]
])
# Etiquetas (y): El siguiente número en la serie
y_train = np.array([40, 50, 60, 70])

# Re-formateamos los datos para que la RNN los entienda (muestras, pasos de tiempo, características)
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))

# --- PASO 2: Construir el Modelo "Cíclico" ---
model = tf.keras.Sequential([
    # Capa RNN con 10 neuronas.
    # Estas neuronas tienen un "bucle" interno (el ciclo).
    tf.keras.layers.SimpleRNN(10, activation='relu', input_shape=(3, 1)),

    # Capa de salida (1 neurona para darnos el número final)
    tf.keras.layers.Dense(1)
])

# --- PASO 3: Compilar y Entrenar ---
model.compile(optimizer='adam', loss='mse') # MSE es ideal para predecir números (regresión)
print("Entrenando la memoria del ciclo...")
model.fit(X_train, y_train, epochs=500, verbose=0)
print("¡Entrenamiento terminado!")

# --- PASO 4: Predicción ---
secuencia_nueva = np.array([70, 80, 90]).reshape((1, 3, 1))
prediccion = model.predict(secuencia_nueva)

print(f"\nSi la secuencia es [70, 80, 90]...")
print(f"La red predice que el siguiente número es: {prediccion[0][0]:.2f}")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Entrenando la memoria del ciclo...
¡Entrenamiento terminado!
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 154ms/step

Si la secuencia es [70, 80, 90]...
La red predice que el siguiente número es: 117.14


In [2]:
import pandas as pd
import numpy as np
import time
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

# --- AQUÍ ESTABA EL ERROR 404 (YA CORREGIDO) ---
# Usamos una URL estable de Kaggle
url = "https://raw.githubusercontent.com/dsindy/kaggle-titanic/master/data/train.csv"

try:
    data = pd.read_csv(url)
    print("✅ Dataset cargado correctamente.")
except Exception as e:
    print(f"❌ Error cargando el dataset: {e}")
    # Detenemos el código si no carga, porque si no dará error después
    raise

# --- PREPROCESAMIENTO ---
# Seleccionar columnas relevantes
data = data[['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare']]

# Llenar edad con la media
imputer = SimpleImputer(strategy='mean')
data['Age'] = imputer.fit_transform(data[['Age']])
data.dropna(inplace=True)

# Convertir variables categóricas (Sex) a numéricas
le = LabelEncoder()
data['Sex'] = le.fit_transform(data['Sex'])

# Separar X (features) y y (target)
X = data.drop('Survived', axis=1)
y = data['Survived']

# Escalar los datos
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split de entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# --- MODELOS (Tus 4 combinaciones) ---
arquitecturas = [
    {"nombre": "Simple", "capas": [8], "desc": "1 Capa (8 neuronas)"},
    {"nombre": "Intermedia", "capas": [32, 16], "desc": "2 Capas (32 y 16)"},
    {"nombre": "Profunda", "capas": [64, 32, 16], "desc": "3 Capas"},
    {"nombre": "Ancha", "capas": [128, 128], "desc": "2 Capas anchas"}
]

resultados_titanic = []

print(f"\n--- INICIANDO ENTRENAMIENTO DE MODELOS ---")

for arq in arquitecturas:
    # Construcción del modelo
    model = Sequential()
    model.add(Dense(arq["capas"][0], input_dim=X_train.shape[1], activation='relu'))

    for neuronas in arq["capas"][1:]:
        model.add(Dense(neuronas, activation='relu'))

    model.add(Dense(1, activation='sigmoid'))
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    # Entrenar
    print(f"Entrenando modelo: {arq['nombre']}...")
    model.fit(X_train, y_train, epochs=50, batch_size=16, verbose=0)

    # Evaluar
    loss, accuracy = model.evaluate(X_test, y_test, verbose=0)

    # Medir tiempo
    start_time = time.time()
    model.predict(X_test, verbose=0)
    end_time = time.time()

    resultados_titanic.append({
        "Modelo": arq["nombre"],
        "Accuracy": round(accuracy * 100, 2),
        "Tiempo (ms)": round((end_time - start_time) * 1000, 2)
    })

# Mostrar tabla final
df_res = pd.DataFrame(resultados_titanic)
print("\n=== RESULTADOS FINALES ===")
print(df_res)

✅ Dataset cargado correctamente.

--- INICIANDO ENTRENAMIENTO DE MODELOS ---
Entrenando modelo: Simple...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Entrenando modelo: Intermedia...
Entrenando modelo: Profunda...


Entrenando modelo: Ancha...



=== RESULTADOS FINALES ===
       Modelo  Accuracy  Tiempo (ms)
0      Simple     81.56       154.46
1  Intermedia     81.56       154.21
2    Profunda     81.56       204.21
3       Ancha     79.89       163.32


In [3]:
import numpy as np
import time
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from sklearn.model_selection import train_test_split

# 1. Generar Dataset Sintético (Oscuro vs Claro)
def generar_dataset(n_muestras=1000, img_size=28):
    X = []
    y = []
    for _ in range(n_muestras):
        es_claro = np.random.randint(0, 2) # 0 o 1
        if es_claro:
            # Generar imagen clara (valores entre 0.6 y 1.0)
            img = np.random.uniform(0.6, 1.0, (img_size, img_size))
            y.append(1)
        else:
            # Generar imagen oscura (valores entre 0.0 y 0.4)
            img = np.random.uniform(0.0, 0.4, (img_size, img_size))
            y.append(0)
        X.append(img)
    return np.array(X), np.array(y)

X_img, y_img = generar_dataset()

# Split de entrenamiento y prueba
X_train_img, X_test_img, y_train_img, y_test_img = train_test_split(X_img, y_img, test_size=0.2, random_state=42)

# --- DEFINICIÓN DE 4 ARQUITECTURAS DIFERENTES ---
# Nota: Como es procesamiento de imagen simple, usaremos Flatten + Dense
arquitecturas_img = [
    {"nombre": "Minimalista", "capas": [4], "desc": "1 Capa (4 neuronas)"},
    {"nombre": "Estándar", "capas": [64, 32], "desc": "2 Capas (64, 32)"},
    {"nombre": "Muy Profunda", "capas": [128, 64, 32, 16], "desc": "4 Capas densas"},
    {"nombre": "Excesiva", "capas": [512, 256], "desc": "2 Capas masivas"}
]

resultados_img = []

print(f"\n--- INICIANDO PRUEBAS IMÁGENES (CLARO VS OSCURO) ---")

for arq in arquitecturas_img:
    model = Sequential()
    model.add(Flatten(input_shape=(28, 28))) # Aplanar la imagen de 28x28 a vector de 784

    for neuronas in arq["capas"]:
        model.add(Dense(neuronas, activation='relu'))

    model.add(Dense(1, activation='sigmoid')) # Salida binaria

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    # Entrenar
    model.fit(X_train_img, y_train_img, epochs=10, batch_size=32, verbose=0)

    # Evaluar
    loss, accuracy = model.evaluate(X_test_img, y_test_img, verbose=0)

    # Medir tiempo
    start_time = time.time()
    model.predict(X_test_img, verbose=0)
    end_time = time.time()
    tiempo_total = (end_time - start_time) * 1000

    resultados_img.append({
        "Modelo": arq["nombre"],
        "Detalle": arq["desc"],
        "Accuracy": round(accuracy * 100, 2),
        "Tiempo (ms)": round(tiempo_total, 2)
    })
    print(f"Modelo {arq['nombre']} completado.")

# Mostrar tabla final
df_res_img = pd.DataFrame(resultados_img)
print("\n=== RESULTADOS IMÁGENES ===")
print(df_res_img)
idx_mejor = df_res_img['Accuracy'].idxmax()
# En caso de empate en accuracy, solemos preferir el modelo más simple (menor tiempo/tamaño)
print(f"\nMejor opción técnica: {df_res_img.loc[idx_mejor, 'Modelo']} con {df_res_img.loc[idx_mejor, 'Accuracy']}% de precisión.")


--- INICIANDO PRUEBAS IMÁGENES (CLARO VS OSCURO) ---


/usr/local/lib/python3.12/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Modelo Minimalista completado.
Modelo Estándar completado.
Modelo Muy Profunda completado.
Modelo Excesiva completado.

=== RESULTADOS IMÁGENES ===
         Modelo              Detalle  Accuracy  Tiempo (ms)
0   Minimalista  1 Capa (4 neuronas)      64.0       149.91
1      Estándar     2 Capas (64, 32)     100.0       168.21
2  Muy Profunda       4 Capas densas     100.0       409.52
3      Excesiva      2 Capas masivas     100.0       188.24

Mejor opción técnica: Estándar con 100.0% de precisión.
